In [1]:
from fastapi_offline import FastAPIOffline
import uvicorn
import asyncio
from fastapi.templating import Jinja2Templates
from fastapi.responses import HTMLResponse
from fastapi import FastAPI, HTTPException, Request

In [2]:
# Database setting up

import sqlite3

def connect_db():
  conn = sqlite3.connect("students.db")
  conn.row_factory = sqlite3.Row
  return conn

# Create database and table if not exists
conn = connect_db()
cur = conn.cursor()

# Create table
cur.execute("""
CREATE TABLE IF NOT EXISTS students (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id VARCHAR(13) NOT NULL,
    name TEXT NOT NULL,
    division TEXT
)
""")
conn.commit()
conn.close()

In [3]:
# CRUD operations

def get_students():
  conn = connect_db()
  cur = conn.cursor()
  students = cur.execute("SELECT * FROM students").fetchall()
  conn.close()
  return [dict(student) for student in students]

def get_student(student_id):
  conn = connect_db()
  cur = conn.cursor()
  student = cur.execute("SELECT * FROM students WHERE student_id = ?", (student_id,)).fetchone()
  conn.close()
  return dict(student) if student else None

def create_student(student_id, name, division):
  conn = connect_db()
  cur = conn.cursor()
  cur.execute("INSERT INTO students (student_id, name, division) VALUES (?, ?, ?)", (student_id, name, division))
  conn.commit()

  # Get the last inserted student ID
  student_id = cur.lastrowid
  conn.close()
  return get_student(student_id)

def update_student(student_id, name, division):
  conn = connect_db()
  cur = conn.cursor()
  cur.execute("UPDATE students SET name = ?, division = ? WHERE student_id = ?", (name, division, student_id))
  conn.commit()
  conn.close()

  return get_student(student_id)

def delete_student(student_id):
  conn = connect_db()
  cur = conn.cursor()
  cur.execute("DELETE FROM students WHERE student_id = ?", (student_id,))
  conn.commit()
  conn.close()

In [13]:
from pydantic import BaseModel
from typing import Optional

class Student(BaseModel):
  student_id: str
  name: str
  division: Optional[str] = None

In [14]:
# FastAPI app

app = FastAPIOffline()
templates = Jinja2Templates(directory="templates")  # Set up Jinja2 templates

# Routing
@app.get("/")
async def root():
    return {"message": "Hello World"}


# Rest API
@app.post("/api/students")
async def create_student_endpoint(student: Student):

    incomming_data = student.model_dump()
    student_id = incomming_data["student_id"]
    name = incomming_data["name"]
    division = incomming_data.get("division")

    student = create_student(student_id, name, division)
    return student

@app.put("/api/students")
async def update_student_endpoint(student: Student):

    incomming_data = student.model_dump()
    student_id = incomming_data["student_id"]
    name = incomming_data["name"]
    division = incomming_data.get("division")

    existing_student = get_student(student_id)
    if not existing_student:
        raise HTTPException(status_code=404, detail="Student not found")

    updated_student = update_student(student_id, name, division)
    return updated_student

@app.delete("/api/students/{student_id}")
async def delete_student_endpoint(student_id: str):
    existing_student = get_student(student_id)
    if not existing_student:
        raise HTTPException(status_code=404, detail="Student not found")

    delete_student(student_id)
    return {"message": "Student deleted successfully"}

In [16]:
# HTML Pages
@app.get("/students")
async def students_html(request: Request):
    return templates.TemplateResponse(
        "students.html",
        {
            "request": request,
            "students": get_students()
        }
    )

@app.get("/students/add")
async def add_student_html(request: Request):
    return templates.TemplateResponse(
        "add_student.html",
        {
            "request": request
        }
    )

@app.get("/students/{student_id}/edit")
async def edit_student_html(request: Request, student_id: str):
    student = get_student(student_id)
    if not student:
        raise HTTPException(status_code=404, detail="Student not found")

    return templates.TemplateResponse(
        "edit_student.html",
        {
            "request": request,
            "student": student
        }
    )

if __name__ == "__main__":
  config = uvicorn.Config(app)
  server = uvicorn.Server(config)
  await server.serve()

INFO:     Started server process [26184]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:59504 - "GET /students HTTP/1.1" 200 OK
INFO:     127.0.0.1:59504 - "GET /student/add HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:59224 - "GET /students HTTP/1.1" 200 OK
INFO:     127.0.0.1:59224 - "GET /students/add HTTP/1.1" 200 OK
INFO:     127.0.0.1:62020 - "POST /api/students HTTP/1.1" 200 OK
INFO:     127.0.0.1:62020 - "GET /students HTTP/1.1" 200 OK
INFO:     127.0.0.1:62020 - "GET /students/6706021411202/edit HTTP/1.1" 200 OK
INFO:     127.0.0.1:62020 - "PUT /api/students HTTP/1.1" 200 OK
INFO:     127.0.0.1:62020 - "GET /students HTTP/1.1" 200 OK
INFO:     127.0.0.1:62020 - "GET /students/add HTTP/1.1" 200 OK
INFO:     127.0.0.1:64323 - "POST /api/students HTTP/1.1" 200 OK
INFO:     127.0.0.1:64323 - "GET /students HTTP/1.1" 200 OK
INFO:     127.0.0.1:64323 - "GET /students/add HTTP/1.1" 200 OK
INFO:     127.0.0.1:54718 - "POST /api/students HTTP/1.1" 200 OK
INFO:     127.0.0.1:54718 - "GET /students HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [26184]
